# 03 Encoding Representations

This notebook runs the 5 encoders specified in the project plan:
1. TF-IDF + SVD (300D)
2. Word2Vec (300D)
3. GloVe (300D)
4. SBERT (384D)
5. BGE-large (1024D)

Embeddings are saved to `data/embeddings/`.

In [ ]:
import pandas as pd
import numpy as np
import os
import sys
sys.path.append('..')

from src.encoders import TFIDFEncoder, W2VEncoder, GloVeEncoder, SBERTEncoder, BGEEncoder
from src.preprocess import preprocess_data

## 1. Load and Preprocess Data

In [ ]:
# Load processed data if available, otherwise load raw and preprocess
processed_path = '../data/processed/cleaned_reviews.parquet'
raw_dir = '../data/raw/'

if os.path.exists(processed_path):
    print("Loading processed data...")
    df = pd.read_parquet(processed_path)
else:
    from glob import glob
    latest_raw = max(glob(os.path.join(raw_dir, "*.jsonl")), key=os.path.getctime)
    print(f"Loading raw data from {latest_raw}...")
    df = pd.read_json(latest_raw, lines=True)
    df = preprocess_data(df)
    
print(f"Data shape: {df.shape}")

## 2. Initialize and Run Encoders

We will save each embedding matrix as a `.npy` file.

In [ ]:
embedding_dir = '../data/embeddings/'
os.makedirs(embedding_dir, exist_ok=True)
texts = df['cleaned_text'].tolist()

### 2.1 TF-IDF + SVD

In [ ]:
tfidf_encoder = TFIDFEncoder(n_components=300)
tfidf_embeddings = tfidf_encoder.encode(texts)
np.save(os.path.join(embedding_dir, 'tfidf.npy'), tfidf_embeddings)
print("Saved TF-IDF embeddings.")

### 2.2 Word2Vec (Trained on Corpus)

In [ ]:
tokenized_texts = [text.split() for text in texts]
w2v_encoder = W2VEncoder(vector_size=300)
w2v_embeddings = w2v_encoder.encode(tokenized_texts)
np.save(os.path.join(embedding_dir, 'w2v.npy'), w2v_embeddings)
print("Saved Word2Vec embeddings.")

### 2.3 GloVe (840B 300D)

> **Note:** Requires `glove.840B.300d.txt` in the root or specified path. If missing, this will skip or error.

In [ ]:
glove_path = '../glove.840B.300d.txt'
if os.path.exists(glove_path):
    glove_encoder = GloVeEncoder(glove_path=glove_path)
    glove_embeddings = glove_encoder.encode(tokenized_texts)
    np.save(os.path.join(embedding_dir, 'glove.npy'), glove_embeddings)
    print("Saved GloVe embeddings.")
else:
    print("GloVe file not found. Skipping.")

### 2.4 SBERT (MiniLM)

In [ ]:
sbert_encoder = SBERTEncoder()
sbert_embeddings = sbert_encoder.encode(texts)
np.save(os.path.join(embedding_dir, 'sbert.npy'), sbert_embeddings)
print("Saved SBERT embeddings.")

### 2.5 BGE-large

In [ ]:
bge_encoder = BGEEncoder()
bge_embeddings = bge_encoder.encode(texts)
np.save(os.path.join(embedding_dir, 'bge.npy'), bge_embeddings)
print("Saved BGE embeddings.")